# เกม หรือ การค้นหาแบบมีคู่ต่อสู้ (GAMES OR ADVERSARIAL SEARCH)

โน้ตบุ๊คนี้ใช้เป็นเนื้อหาสนับสนุนสำหรับหัวข้อที่ครอบคลุมใน **บทที่ 5 - การค้นหาแบบมีคู่ต่อสู้** ในหนังสือ *Artificial Intelligence: A Modern Approach.* โน้ตบุ๊คนี้ใช้การดำเนินงานจากโมดูล [games.py](https://github.com/aimacode/aima-python/blob/master/games.py) มาเรียกใช้ class, method, ตัวแปรสากล ฯลฯ ที่ต้องการจากโมดูล games

# สารบัญ (CONTENTS)

* การแทนค่าเกม (Game Representation)
* ตัวอย่างเกม (Game Examples)
    * Tic-Tac-Toe
    * เกมรูปที่ 5.2 (Figure 5.2 Game)
* Min-Max
* Alpha-Beta
* ผู้เล่น (Players)
* มาเล่นเกมกัน! (Let's Play Some Games!)

In [1]:
from games import *
from notebook import psource, pseudocode

# การแทนค่าเกม (GAME REPRESENTATION)

ในการแทนค่าเกม เราจะใช้ class `Game` ซึ่งเราสามารถสร้าง subclass และเขียนฟังก์ชันของมันใหม่เพื่อแทนค่าเกมของเราเอง เครื่องมือช่วยคือ namedtuple `GameState` ซึ่งในบางกรณีจะมีประโยชน์มาก โดยเฉพาะเมื่อเกมของเราต้องการให้เราจำกระดาน (เช่น หมากรุก)

## namedtuple `GameState`

`GameState` เป็น [namedtuple](https://docs.python.org/3.5/library/collections.html#collections.namedtuple) ที่แทนค่าสถานะปัจจุบันของเกม มันถูกใช้เพื่อช่วยแทนค่าเกมที่มีสถานะไม่สามารถแทนค่าได้ง่ายๆ แบบปกติ หรือสำหรับเกมที่ต้องการความจำของกระดาน เช่น Tic-Tac-Toe

`Gamestate` ถูกกำหนดดังนี้:

`GameState = namedtuple('GameState', 'to_move, utility, board, moves')`

* `to_move`: แทนค่าคิวของใครที่จะเดินต่อไป

* `utility`: เก็บค่าประโยชน์ของสถานะเกม การเก็บค่าประโยชน์นี้เป็นไอเดียที่ดี เพราะเมื่อคุณทำ Minimax Search หรือ Alphabeta Search คุณจะสร้างการเรียกซ้ำหลายครั้ง ซึ่งจะเดินทางไปจนถึงสถานะสิ้นสุด เมื่อการเรียกซ้ำเหล่านี้กลับขึ้นไปยังผู้เรียกเดิม เราได้คำนวณค่าประโยชน์สำหรับสถานะเกมหลายสถานะแล้ว เราจึงเก็บค่าประโยชน์เหล่านี้ใน `GameState` ที่เกี่ยวข้องเพื่อหลีกเลี่ยงการคำนวณซ้ำทั้งหมดอีกครั้ง

* `board`: dict ที่เก็บกระดานของเกม

* `moves`: เก็บรายการของการเดินที่ถูกกฎหมายที่เป็นไปได้จากตำแหน่งปัจจุบัน

## class `Game`

มาดู class `Game` ในโมดูลของเราเรา เราจะเห็นว่ามันมีฟังก์ชัน ได้แก่ `actions`, `result`, `utility`, `terminal_test`, `to_move` และ `display`

เราจะเห็นว่าฟังก์ชันเหล่านี้ยังไม่ได้ถูกดำเนินการจริง class นี้เป็นเพียง template class เรากำลังจะสร้าง class สำหรับเกมของเรา โดยการสืบทอด class `Game` นี้และการดำเนินการ method ทั้งหมดที่กล่าวถึงใน `Game`

In [2]:
%psource Game

class Game:
    """A game is similar to a problem, but it has a utility for each
    state and a terminal test instead of a path cost and a goal
    test. To create a game, subclass this class and implement actions,
    result, utility, and terminal_test. You may override display and
    successors or you can inherit their default methods. You will also
    need to set the .initial attribute to the initial state; this can
    be done in the constructor."""

    def actions(self, state):
        """Return a list of the allowable moves at this point."""
        raise NotImplementedError

    def result(self, state, move):
        """Return the state that results from making a move from a state."""
        raise NotImplementedError

    def utility(self, state, player):
        """Return the value of this final state to player."""
        raise NotImplementedError

    def terminal_test(self, state):
        """Return True if this is a final state for the game."""
        return not self.

ตอนนี้มาดูรายละเอียดของ method ทั้งหมดใน class `Game` ของเรา คุณต้องดำเนินการ method เหล่านี้เมื่อคุณสร้าง class ใหม่ที่จะแทนค่าเกมของคุณ

* `actions(self, state)`: เมื่อได้รับสถานะเกม method นี้จะสร้างการกระทำที่ถูกกฎหมายทั้งหมดที่เป็นไปได้จากสถานะนี้ เป็น list หรือ generator การส่งคืน generator แทน list มีข้อดีคือประหยัดพื้นที่และคุณยังสามารถดำเนินการกับมันเหมือน list


* `result(self, state, move)`: เมื่อได้รับสถานะเกมและการเดิน method นี้จะส่งคืนสถานะเกมที่คุณได้จากการเดินในสถานะเกมนี้


* `utility(self, state, player)`: เมื่อได้รับสถานะเกมสิ้นสุดและผู้เล่น method นี้จะส่งคืนค่าประโยชน์สำหรับผู้เล่นนั้นในสถานะเกมสิ้นสุดที่กำหนด ขณะที่ดำเนินการ method นี้ ให้สมมติว่าสถานะเกมเป็นสถานะเกมสิ้นสุด ตรรกะในโมดูลนี้เป็นเช่นนั้น method นี้จะถูกเรียกเฉพาะในสถานะเกมสิ้นสุด


* `terminal_test(self, state)`: เมื่อได้รับสถานะเกม method นี้ควรส่งคืน `True` ถ้าสถานะเกมนี้เป็นสถานะสิ้นสุด และ `False` ในกรณีอื่น


* `to_move(self, state)`: เมื่อได้รับสถานะเกม method นี้จะส่งคืนผู้เล่นที่จะเล่นต่อไป ข้อมูลนี้มักจะถูกเก็บไว้ในสถานะเกม ดังนั้น method นี้ทำหน้าที่เพียงแยกข้อมูลนี้และส่งคืน


* `display(self, state)`: method นี้พิมพ์/แสดงสถานะปัจจุบันของเกม

# ตัวอย่างเกม (GAME EXAMPLES)

ด้านล่างเราจะให้ตัวอย่างสำหรับเกมที่คุณสามารถสร้างและทดลองใช้งานได้

## Tic-Tac-Toe

ดู class `TicTacToe` method ทั้งหมดที่กล่าวถึงใน class `Game` ได้ถูกดำเนินการที่นี่

In [3]:
%psource TicTacToe

class TicTacToe(Game):
    """Play TicTacToe on an h x v board, with Max (first player) playing 'X'.
    A state has the player to move, a cached utility, a list of moves in
    the form of a list of (x, y) positions, and a board, in the form of
    a dict of {(x, y): Player} entries, where Player is 'X' or 'O'."""

    def __init__(self, h=3, v=3, k=3):
        self.h = h
        self.v = v
        self.k = k
        moves = [(x, y) for x in range(1, h + 1)
                 for y in range(1, v + 1)]
        self.initial = GameState(to_move='X', utility=0, board={}, moves=moves)

    def actions(self, state):
        """Legal moves are any square not yet taken."""
        return state.moves

    def result(self, state, move):
        if move not in state.moves:
            return state  # Illegal move has no effect
        board = state.board.copy()
        board[move] = state.to_move
        moves = list(state.moves)
        moves.remove(move)
        return GameState(to_move=('O' if 

class `TicTacToe` ได้รับการสืบทอดมาจาก class `Game` ดังที่กล่าวไว้ก่อนหน้า คุณจริงๆ อยากจะทำแบบนี้ การตรวจจับบั๊กและข้อผิดพลาดจะง่ายขึ้นมาก

method เพิ่มเติมใน TicTacToe:

* `__init__(self, h=3, v=3, k=3)`: เมื่อคุณสร้าง class ที่สืบทอดมาจาก class `Game` (class `TicTacToe` ในกรณีของเรา) คุณจะต้องสร้าง object ของ class ที่สืบทอดนี้เพื่อเริ่มต้นเกม การเริ่มต้นนี้อาจต้องการข้อมูลเพิ่มเติมบางอย่างที่จะถูกส่งไปยัง `__init__` เป็นตัวแปร สำหรับกรณีของเกม `TicTacToe` ของเรา ข้อมูลเพิ่มเติมนี้จะเป็นจำนวนแถว `h` จำนวนคอลัมน์ `v` และจำนวน X หรือ O ที่ต่อเนื่องกันในแถว คอลัมน์ หรือแนวทแยงที่ต้องการเพื่อชนะ `k` นอกจากนี้ สถานะเกมเริ่มต้นต้องถูกกำหนดที่นี่ใน `__init__`


* `compute_utility(self, board, move, player)`: method ในการคำนวณค่าประโยชน์ของเกม TicTacToe หาก 'X' ชนะด้วยการเดินนี้ method นี้จะส่งคืน 1; หาก 'O' ชนะจะส่งคืน -1; อื่นๆ ส่งคืน 0


* `k_in_row(self, board, move, player, delta_x_y)`: method นี้จะส่งคืน `True` หากมีการสร้างเส้นบนกระดาน TicTacToe ด้วยการเดินล่าสุด มิฉะนั้น `False`

### TicTacToe GameState

ตอนนี้ ก่อนที่เราจะเริ่มดำเนินการเกม `TicTacToe` เราต้องตัดสินใจว่าเราจะแทนค่าสถานะเกมของเราอย่างไร โดยทั่วไป สถานะเกมจะให้ข้อมูลปัจจุบันทั้งหมดเกี่ยวกับเกมในเวลาใดก็ได้ เมื่อคุณได้รับสถานะเกม คุณควรจะสามารถบอกได้ว่าถึงคิวใครต่อไป เกมจะมีลักษณะอย่างไรบนกระดานจริง (หากมี) ฯลฯ สถานะเกมไม่จำเป็นต้องรวมประวัติของเกม หากคุณสามารถเล่นเกมต่อไปได้เมื่อได้รับสถานะเกม การแทนค่าสถานะเกมของคุณก็เป็นที่ยอมรับได้ แม้ว่าเราอาจต้องการรวมข้อมูลทุกประเภทในสถานะเกมของเรา แต่เราไม่อยากใส่ข้อมูลมากเกินไปเข้าไป การปรับเปลี่ยนสถานะเกมนี้เพื่อสร้างสถานะใหม่จะเป็นเรื่องเจ็บปวดจริงๆ

ตอนนี้ สำหรับสถานะเกม `TicTacToe` ของเรา การเก็บเฉพาะตำแหน่งของ X และ O ทั้งหมดจะเพียงพอที่จะแทนค่าข้อมูลเกมทั้งหมดในเวลานั้นหรือไม่? มันบอกเราว่าถึงคิวใครต่อไปหรือไม่? การดู X และ O บนกระดานและนับพวกมันน่าจะบอกเราได้ แต่นั่นหมายถึงการคำนวณเพิ่มเติม เพื่อหลีกเลี่ยงสิ่งนี้ เราจะเก็บว่าเป็นคิวของใครที่จะเดินต่อไปในสถานะเกมด้วย

คิดเกี่ยวกับสิ่งที่เราทำที่นี่ เราได้ลดการคำนวณพิเศษโดยการเก็บข้อมูลเพิ่มเติมในสถานะเกม ตอนนี้ ข้อมูลนี้อาจจะไม่จำเป็นอย่างยิ่งที่จะบอกเราเกี่ยวกับสถานะของเกม แต่มันช่วยประหยัดเวลาการคำนวณเพิ่มเติม เราจะทำเช่นนี้อีกในภายหลัง

ในการเก็บสถานะเกมเราจะใช้ namedtuple `GameState`

* `to_move`: string ของตัวอักษรตัวเดียว ไม่ว่าจะเป็น 'X' หรือ 'O'

* `utility`: 1 สำหรับชนะ, -1 สำหรับแพ้, 0 ในกรณีอื่น

* `board`: ตำแหน่งทั้งหมดของ X และ O บนกระดาน

* `moves`: การเดินทั้งหมดที่เป็นไปได้จากสถานะปัจจุบัน โปรดสังเกตที่นี่ว่า การเก็บการเดินเป็น list ดังที่ทำที่นี่ จะเพิ่มความซับซ้อนของพื้นที่ของ Minimax Search จาก `O(m)` เป็น `O(bm)` อ้างอิงถึงส่วน 5.2.1 ของหนังสือ

### การแทนค่าการเดินในเกม TicTacToe

ตอนนี้ที่เราได้ตัดสินใจแล้วว่าสถานะเกมของเราจะถูกแทนค่าอย่างไร ก็ถึงเวลาตัดสินใจว่าการเดินของเราจะถูกแทนค่าอย่างไร มันง่ายในการใช้การเดินนี้เพื่อปรับเปลี่ยนสถานะเกมปัจจุบันเพื่อสร้างสถานะใหม่

สำหรับเกม `TicTacToe` ของเรา เราจะแทนค่าการเดินด้วย tuple เพียงอันเดียว โดยที่องค์ประกอบแรกและที่สองของ tuple จะแทนค่าแถวและคอลัมน์ตามลำดับ ที่การเดินต่อไปจะถูกทำ ว่าจะทำ 'X' หรือ 'O' จะถูกตัดสินโดย `to_move` ใน namedtuple `GameState`

## เกมรูปที่ 5.2 (Fig52 Game)

สำหรับตัวอย่างที่ง่ายกว่า เราจะแทนค่าเกมใน **รูปที่ 5.2** ของหนังสือ

<img src="images/fig_5_2.png" width="75%">

สถานะถูกแทนค่าด้วยตัวอักษรพิมพ์ใหญ่ภายในสามเหลี่ยม (เช่น "A") ในขณะที่การเดินคือป้ายบนขอบระหว่างสถานะ (เช่น "a1") โหนดสิ้นสุดมีค่าประโยชน์ โปรดสังเกตว่าโหนดสิ้นสุดจะถูกตั้งชื่อในตัวอย่างนี้เป็น 'B1', 'B2' และ 'B3' สำหรับโหนดด้านล่าง 'B' และอื่นๆ

เราจะสร้างแบบจำลองการเดิน ค่าประโยชน์ และสถานะเริ่มต้นแบบนี้:

In [ ]:
moves = dict(A=dict(a1='B', a2='C', a3='D'),
                 B=dict(b1='B1', b2='B2', b3='B3'),
                 C=dict(c1='C1', c2='C2', c3='C3'),
                 D=dict(d1='D1', d2='D2', d3='D3'))
utils = dict(B1=3, B2=12, B3=8, C1=2, C2=4, C3=6, D1=14, D2=5, D3=2)
initial = 'A'

ใน `moves` เรามีระบบ dictionary ที่ซ้อนกัน dictionary ภายนอกมี key เป็นสถานะและ value เป็นการเดินที่เป็นไปได้จากสถานะนั้น (เป็น dictionary) dictionary ภายในของการเดินมี key เป็นชื่อการเดินและ value เป็นสถานะถัดไปหลังจากการเดินเสร็จสิ้น

ด้านล่างเป็นตัวอย่างที่แสดง `moves` เราต้องการสถานะถัดไปหลังจากการเดิน 'a1' จาก 'A' ซึ่งคือ 'B' การดูภาพข้างบนอย่างรวดเร็วจะยืนยันว่านี่คือกรณีจริง

In [ ]:
print(moves['A']['a1'])

ตอนนี้เราจะดูฟังก์ชันที่เราต้องดำเนินการ ก่อนอื่นเราต้องสร้าง object ของ class `Fig52Game`

In [ ]:
fig52 = Fig52Game()

`actions`: ส่งคืนรายการการเดินที่สามารถทำได้จากสถานะที่กำหนด

In [ ]:
psource(Fig52Game.actions)

In [ ]:
print(fig52.actions('B'))

`result`: ส่งคืนสถานะถัดไปหลังจากที่เราทำการเดินเฉพาะ

In [ ]:
psource(Fig52Game.result)

In [ ]:
print(fig52.result('A', 'a1'))

`utility`: ส่งคืนค่าของสถานะสิ้นสุดสำหรับผู้เล่น ('MAX' และ 'MIN') โปรดสังเกตว่าสำหรับ 'MIN' ค่าที่ส่งคืนเป็นค่าลบของค่าประโยชน์

In [ ]:
psource(Fig52Game.utility)

In [ ]:
print(fig52.utility('B1', 'MAX'))
print(fig52.utility('B1', 'MIN'))

`terminal_test`: ส่งคืน `True` ถ้าสถานะที่กำหนดเป็นสถานะสิ้นสุด มิฉะนั้น `False`

In [ ]:
psource(Fig52Game.terminal_test)

In [ ]:
print(fig52.terminal_test('C3'))

`to_move`: ส่งคืนผู้เล่นที่จะเดินในสถานะนี้

In [ ]:
psource(Fig52Game.to_move)

In [ ]:
print(fig52.to_move('A'))

โดยรวมแล้ว class `Fig52` ที่สืบทอดมาจาก class `Game` และเขียนฟังก์ชันของมันใหม่:

In [ ]:
psource(Fig52Game)

# MIN-MAX

## ภาพรวม (Overview)

อัลกอริทึมนี้ (มักเรียกว่า *Minimax*) คำนวณการเดินต่อไปสำหรับผู้เล่น (MIN หรือ MAX) ในสถานะปัจจุบันของพวกเขา มันคำนวณค่า minimax ของสถานะที่ตามมาแบบเรียกซ้ำ จนกว่าจะถึงสถานะสิ้นสุด (ใบของต้นไม้) โดยใช้ค่า `utility` ของสถานะสิ้นสุด มันคำนวณค่าของสถานะพาเรนต์จนกว่าจะถึงโหนดเริ่มต้น (รากของต้นไม้)

เป็นที่น่าสังเกตว่าอัลกอริทึมทำงานแบบ depth-first สามารถดู pseudocode ได้ด้านล่าง:

In [ ]:
pseudocode("Minimax-Decision")

## การดำเนินงาน (Implementation)

ในการดำเนินงานเรากำลังใช้สองฟังก์ชัน `max_value` และ `min_value` เพื่อคำนวณการเดินที่ดีที่สุดสำหรับ MAX และ MIN ตามลำดับ ฟังก์ชันเหล่านี้โต้ตอบในการเรียกซ้ำสลับกัน หนึ่งเรียกอีกหนึ่งจนกว่าจะถึงสถานะสิ้นสุด เมื่อการเรียกซ้ำหยุด เราจะเหลือคะแนนสำหรับการเดินแต่ละครั้ง เราส่งคืนค่าสูงสุด แม้ว่าจะส่งคืนค่าสูงสุด มันจะทำงานสำหรับ MIN ด้วย เพราะสำหรับ MIN ค่าเหล่านั้นเป็นค่าลบของพวกมัน (ดังนั้นลำดับของค่าจะกลับด้าน ดังนั้นยิ่งสูงยิ่งดีสำหรับ MIN ด้วย)

In [ ]:
psource(minimax_decision)

## ตัวอย่าง (Example)

ตอนนี้เราจะเล่นเกม Fig52 โดยใช้อัลกอริทึมนี้ ดูเกม Fig52Game จากข้างบนเพื่อติดตาม

ถึงคิว MAX ที่จะเดิน และเขาอยู่ในสถานะ A เขาสามารถไปยัง B, C หรือ D โดยใช้การเดิน a1, a2 และ a3 ตามลำดับ เป้าหมายของ MAX คือการเพิ่มค่าสุดท้ายให้ได้มากที่สุด ดังนั้น เพื่อตัดสินใจ MAX ต้องการทราบค่าที่โหนดดังกล่าวและเลือกค่าที่มากที่สุด หลังจาก MAX ถึงคิว MIN ที่จะเล่น ดังนั้น MAX ต้องการทราบว่าค่าของ B, C และ D จะเป็นเท่าใดหลังจาก MIN เล่น

ปัญหาจะกลายเป็นการเดินใดที่ MIN จะทำที่ B, C และ D สถานะตัวตามของโหนดทั้งหมดเหล่านี้เป็นสถานะสิ้นสุด ดังนั้น MIN จะเลือกค่าที่เล็กที่สุดสำหรับแต่ละโหนด ดังนั้น สำหรับ B เขาจะเลือก 3 (จากการเดิน b1), สำหรับ C เขาจะเลือก 2 (จากการเดิน c1) และสำหรับ D เขาจะเลือก 2 อีกครั้ง (จากการเดิน d3)

มาดูในโค้ด:

In [ ]:
print(minimax_decision('B', fig52))
print(minimax_decision('C', fig52))
print(minimax_decision('D', fig52))

ตอนนี้ MAX ทราบว่าค่าสำหรับ B, C และ D คือ 3, 2 และ 2 (ผลิตโดยการเดินข้างบนของ MIN) ค่าที่มากที่สุดคือ 3 ซึ่งเขาจะได้ด้วยการเดิน a1 นี่คือการเดินที่ MAX จะทำ มาดูอัลกอริทึมในการกระทำเต็มรูปแบบ:

In [ ]:
print(minimax_decision('A', fig52))

## การแสดงผลด้วยภาพ (Visualization)

ด้านล่างเรามีการแสดงผลเกมง่ายๆ โดยใช้อัลกอริทึม หลังจากที่คุณรันคำสั่ง ให้คลิกที่เซลล์เพื่อเดินเกมต่อไป คุณสามารถป้อนค่าของคุณเองผ่านรายการของจำนวนเต็ม 27 ตัว

In [ ]:
from notebook import Canvas_minimax
from random import randint

In [ ]:
minimax_viz = Canvas_minimax('minimax_viz', [randint(1, 50) for i in range(27)])

# ALPHA-BETA

## ภาพรวม (Overview)

ในขณะที่ *Minimax* เหมาะสำหรับการคำนวณการเดิน แต่มันสามารถกลายเป็นเรื่องยุ่งยากเมื่อจำนวนสถานะเกมมีมากขึ้น อัลกอริทึมต้องค้นหาใบทั้งหมดของต้นไม้ ซึ่งเพิ่มขึ้นแบบเลขชี้กำลังกับความลึกของมัน

สำหรับ Tic-Tac-Toe ที่มีความลึกของต้นไม้คือ 9 (หลังจากการเดินที่ 9 เกมจบ) เราสามารถมีสถานะสิ้นสุดได้มากที่สุด 9! (มากที่สุดเพราะไม่ใช่โหนดสิ้นสุดทั้งหมดอยู่ที่ระดับสุดท้ายของต้นไม้; บางตัวอยู่สูงขึ้นเพราะเกมจบก่อนการเดินที่ 9) นี่ไม่เลวเท่าไหร่ แต่สำหรับปัญหาที่ซับซ้อนกว่าเช่นหมากรุก เรามีโหนดสิ้นสุดมากกว่า $10^{40}$ โชคไม่ดีที่เรายังไม่พบวิธีตัดเลขชี้กำลังออกไป แต่เราพบวิธีบรรเทาภาระงาน

ที่นี่เราจะตรวจสอบการ *ตัดแต่ง (pruning)* ต้นไม้เกม ซึ่งหมายถึงการเอาส่วนที่เราไม่จำเป็นต้องตรวจสอบออกไป ประเภทของการตัดแต่งเฉพาะนี้เรียกว่า *alpha-beta* และการค้นหาโดยรวมเรียกว่า *alpha-beta search*

เพื่อแสดงส่วนใดของต้นไม้ที่เราไม่จำเป็นต้องค้นหา เราจะดูตัวอย่าง `Fig52Game`

ในตัวอย่างเกม เราต้องหาการเดินที่ดีที่สุดสำหรับผู้เล่น MAX ที่สถานะ A ซึ่งคือค่าสูงสุดของการเดินที่เป็นไปได้ของ MIN ที่สถานะตัวตาม

`MAX(A) = MAX( MIN(B), MIN(C), MIN(D) )`

`MIN(B)` คือค่าต่ำสุดของ 3, 12, 8 ซึ่งคือ 3 ดังนั้นสูตรข้างบนจะกลายเป็น:

`MAX(A) = MAX( 3, MIN(C), MIN(D) )`

การเดินต่อไปที่เราจะตรวจสอบคือ c1 ซึ่งนำไปสู่สถานะสิ้นสุดด้วยค่าประโยชน์ 2 ก่อนที่เราจะค้นหาต่อภายใต้สถานะ C มากลับไปในสูตรของเราด้วยค่าใหม่:

`MAX(A) = MAX( 3, MIN(2, c2, .... cN), MIN(D) )`

เราไม่ทราบว่าสถานะ C อนุญาตการเดินกี่ครั้ง แต่เราทราบว่าการเดินแรกได้ผลลัพธ์ค่า 2 เราต้องค้นหาต่อภายใต้ C หรือไม่? คำตอบคือไม่ ค่าที่ MIN จะเลือกใน C จะมากที่สุด 2 เนื่องจาก MAX มีตัวเลือกที่จะเลือกบางอย่างที่มากกว่านั้นแล้ว คือ 3 จาก B เขาไม่จำเป็นต้องค้นหาต่อภายใต้ C

ใน *alpha-beta* เราใช้พารามิเตอร์เพิ่มเติมสองตัวสำหรับแต่ละสถานะ/โหนด *a* และ *b* ที่อธิบายขอบเขตในการเดินที่เป็นไปได้ พารามิเตอร์ *a* แสดงตัวเลือกที่ดีที่สุด (ค่าสูงสุด) สำหรับ MAX ตามเส้นทางนั้น ในขณะที่ *b* แสดงตัวเลือกที่ดีที่สุด (ค่าต่ำสุด) สำหรับ MIN เมื่อเราดำเนินไป เราจะอัปเดต *a* และ *b* และตัดแต่งแขนงโหนดเมื่อค่าของโหนดแย่กว่าค่าของ *a* และ *b* สำหรับ MAX และ MIN ตามลำดับ

ในตัวอย่างข้างบน หลังจากการค้นหาภายใต้สถานะ B, MAX มีค่า *a* เป็น 3 ดังนั้น เมื่อค้นหาโหนด C เราพบค่าน้อยกว่านั้น คือ 2 เราจึงหยุดการค้นหาภายใต้ C

คุณสามารถอ่าน pseudocode ด้านล่าง:

In [ ]:
pseudocode("Alpha-Beta-Search")

## การดำเนินงาน (Implementation)

เช่นเดียวกับ *minimax* เราใช้ฟังก์ชัน `max_value` และ `min_value` อีกครั้ง แต่คราวนี้เราใช้ค่า *a* และ *b* อัปเดตพวกมันและหยุดการเรียกแบบเรียกซ้ำหากเราไปสู่โหนดที่มีค่าแย่กว่า *a* และ *b* (สำหรับ MAX และ MIN) อัลกอริทึมหาค่าสูงสุดและส่งคืนการเดินที่ได้ผลลัพธ์นั้น

การดำเนินงาน:

In [ ]:
%psource alphabeta_search

## ตัวอย่าง (Example)

เราจะเล่นเกม Fig52 ด้วยอัลกอริทึม *alpha-beta* search ถึงคิว MAX ที่จะเล่นที่สถานะ A

In [ ]:
print(alphabeta_search('A', fig52))

การเดินที่ดีที่สุดสำหรับ MAX คือ a1 ด้วยเหตุผลที่ให้ไว้ข้างบน MIN จะเลือกการเดิน b1 สำหรับ B ได้ผลลัพธ์ค่า 3 อัปเดตค่า *a* ของ MAX เป็น 3 จากนั้น เมื่อเราพบภายใต้ C โหนดที่มีค่า 2 เราจะหยุดการค้นหาภายใต้ sub-tree นั้นเนื่องจากมันน้อยกว่า *a* จาก D เรามีค่า 2 ดังนั้น การเดินที่ดีที่สุดสำหรับ MAX คือการที่ได้ผลลัพธ์ค่า 3 ซึ่งคือ a1

ด้านล่างเราเห็นการเดินที่ดีที่สุดสำหรับ MIN เริ่มจาก B, C และ D ตามลำดับ โปรดสังเกตว่าอัลกอริทึมในกรณีเหล่านี้ทำงานเหมือนกับ *minimax* เนื่องจากโหนดทั้งหมดด้านล่างสถานะที่กล่าวถึงข้างบนเป็นสถานะสิ้นสุด

In [ ]:
print(alphabeta_search('B', fig52))
print(alphabeta_search('C', fig52))
print(alphabeta_search('D', fig52))

## การแสดงผลด้วยภาพ (Visualization)

ด้านล่างคุณจะพบการแสดงผลด้วยภาพของอัลกอริทึม alpha-beta สำหรับเกมง่ายๆ คลิกที่เซลล์หลังจากที่คุณรันคำสั่งเพื่อเดินเกมต่อไป คุณสามารถป้อนค่าของคุณเองผ่านรายการของจำนวนเต็ม 27 ตัว

In [ ]:
from notebook import Canvas_alphabeta
from random import randint

In [ ]:
alphabeta_viz = Canvas_alphabeta('alphabeta_viz', [randint(1, 50) for i in range(27)])

# ผู้เล่น (PLAYERS)

ดังนั้น เราจึงจบการดำเนินงานของ class `TicTacToe` และ `Fig52Game` สิ่งที่ class เหล่านี้ทำคือการกำหนดกฎของเกม เราต้องการมากกว่านั้นเพื่อสร้าง AI ที่สามารถเล่นเกมได้จริง นี่คือจุดที่ `random_player` และ `alphabeta_player` เข้ามา

## query_player
ฟังก์ชัน `query_player` ช่วยให้คุณ คู่ต่อสู้มนุษย์ เล่นเกมได้ ฟังก์ชันนี้ต้องการ method `display` ที่จะถูกดำเนินการใน game class ของคุณ เพื่อให้สถานะเกมที่ต่อเนื่องกันสามารถแสดงบนเทอร์มินัล ทำให้ง่ายต่อการมองเห็นเกมและเล่นตามนั้น

## random_player
`random_player` เป็นฟังก์ชันที่เล่นการเดินแบบสุ่มในเกม นั่นคือทั้งหมด ไม่มีอะไรมากไปกว่านี้สำหรับตัวนี้

## alphabeta_player
`alphabeta_player` ในทางกลับกัน เรียกฟังก์ชัน `alphabeta_search` ซึ่งส่งคืนการเดินที่ดีที่สุดในสถานะเกมปัจจุบัน ดังนั้น `alphabeta_player` จึงเล่นการเดินที่ดีที่สุดเสมอเมื่อได้รับสถานะเกม สมมติว่าต้นไม้เกมมีขนาดเล็กพอที่จะค้นหาทั้งหมด

## play_game
ฟังก์ชัน `play_game` จะเป็นตัวที่จะถูกใช้จริงในการเล่นเกม คุณส่งอาร์กิวเมนต์ให้มันเป็นตัวอย่างของเกมที่คุณต้องการเล่นและผู้เล่นที่คุณต้องการในเกมนี้ ใช้มันเพื่อเล่น AI vs AI, AI vs มนุษย์ หรือแม้แต่มนุษย์ vs มนุษย์!

# มาเล่นเกมกัน! (LET'S PLAY SOME GAMES!)

## Game52

มาเริ่มต้นด้วยการทดลองกับ `Fig52Game` ก่อน สำหรับนั้นเราจะสร้างตัวอย่างของ subclass Fig52Game ที่สืบทอดมาจาก class Game:

In [ ]:
game52 = Fig52Game()

ก่อนอื่นเราลองใช้ `random_player(game, state)` ของเรา เมื่อได้รับสถานะเกม มันจะให้การเดินแบบสุ่มทุกครั้ง:

In [ ]:
print(random_player(game52, 'A'))
print(random_player(game52, 'A'))

`alphabeta_player(game, state)` จะให้การเดินที่ดีที่สุดที่เป็นไปได้เสมอ สำหรับผู้เล่นที่เกี่ยวข้อง (MAX หรือ MIN):

In [ ]:
print( alphabeta_player(game52, 'A') )
print( alphabeta_player(game52, 'B') )
print( alphabeta_player(game52, 'C') )

สิ่งที่ `alphabeta_player` ทำคือ มันเรียก method `alphabeta_full_search` อย่างง่ายๆ ทั้งสองเป็นอย่างเดียวกันโดยพื้นฐาน ในโมดูล ทั้ง `alphabeta_full_search` และ `minimax_decision` ได้ถูกดำเนินการ ทั้งสองทำงานเดียวกันและส่งคืนสิ่งเดียวกัน ซึ่งคือการเดินที่ดีที่สุดในสถานะปัจจุบัน แค่ว่า `alphabeta_full_search` มีประสิทธิภาพมากกว่าในเรื่องเวลาเพราะมันตัดแต่งต้นไม้ค้นหาและด้วยเหตุนี้ สำรวจสถานะได้น้อยกว่า

In [ ]:
minimax_decision('A', game52)

In [ ]:
alphabeta_search('A', game52)

สาธิตฟังก์ชัน play_game กับ game52:

In [ ]:
game52.play_game(alphabeta_player, alphabeta_player)

In [ ]:
game52.play_game(alphabeta_player, random_player)

In [ ]:
game52.play_game(query_player, alphabeta_player)

In [ ]:
game52.play_game(alphabeta_player, query_player)

โปรดสังเกตว่าหากคุณเป็นผู้เล่นคนแรก alphabeta_player จะเล่นเป็น MIN และหากคุณเป็นผู้เล่นคนที่สอง alphabeta_player จะเล่นเป็น MAX นี่เกิดขึ้นเพราะนั่นคือวิธีที่เกมถูกกำหนดใน class Fig52Game การดูโค้ดของ class นี้ควรจะทำให้ชัดเจน

## TicTacToe

ตอนนี้มาเล่น `TicTacToe` ก่อนอื่นเราเริ่มต้นเกมโดยการสร้างตัวอย่างของ subclass TicTacToe ที่สืบทอดมาจาก class Game:

In [ ]:
ttt = TicTacToe()

เราสามารถพิมพ์สถานะโดยใช้ method display:

In [ ]:
ttt.display(ttt.initial)

อืม นั่นคือสถานะเริ่มต้นของเกม ไม่มี X และไม่มี O

มาสร้างสถานะเกมใหม่ด้วยตัวเราเองเพื่อทดลอง:

In [ ]:
my_state = GameState(
    to_move = 'X',
    utility = '0',
    board = {(1,1): 'X', (1,2): 'O', (1,3): 'X',
             (2,1): 'O',             (2,3): 'O',
             (3,1): 'X',
            },
    moves = [(2,2), (3,2), (3,3)]
    )

แล้วสถานะเกมนี้มีลักษณะอย่างไร?

In [ ]:
ttt.display(my_state)

`random_player` จะทำตัวอย่างที่เขาควรจะเป็น กล่าวคือ *แบบสุ่มเทียม*:

In [ ]:
random_player(ttt, my_state)

In [ ]:
random_player(ttt, my_state)

แต่ `alphabeta_player` จะให้การเดินที่ดีที่สุดเสมอ ตามที่คาดหวัง:

In [ ]:
alphabeta_player(ttt, my_state)

ตอนนี้มาทำให้ผู้เล่นสองคนเล่นต่อกัน เราใช้ฟังก์ชัน `play_game` สำหรับสิ่งนี้ ฟังก์ชัน `play_game` ทำให้ผู้เล่นเล่นแมตช์ต่อกันและส่งคืนค่าประโยชน์สำหรับผู้เล่นแรกของสถานะสิ้นสุดที่ถึงเมื่อเกมจบ ดังนั้น สำหรับเกม `TicTacToe` ของเรา หากเราได้ผลลัพธ์ +1 ผู้เล่นคนแรกชนะ, -1 หากผู้เล่นคนที่สองชนะ และ 0 หากแมตช์จบลงด้วยการเสมอ

In [ ]:
ttt.play_game(random_player, alphabeta_player)

ผลลัพธ์คือ (โดยปกติ) -1 เพราะ `random_player` แพ้ `alphabeta_player` อย่างไรก็ตาม บางครั้ง `random_player` สามารถเสมอกับ `alphabeta_player` ได้

เนื่องจาก `alphabeta_player` เล่นอย่างสมบูรณ์แบบ แมตช์ระหว่าง `alphabeta_player` สองตัวควรจบลงด้วยการเสมอเสมอ มาดูว่าสิ่งนี้เกิดขึ้นหรือไม่:

In [ ]:
for _ in range(10):
    print(ttt.play_game(alphabeta_player, alphabeta_player))

`random_player` ไม่ควรชนะ `alphabeta_player` เลย มาทดสอบเรื่องนั้น

In [ ]:
for _ in range(10):
    print(ttt.play_game(random_player, alphabeta_player))

## Canvas_TicTacToe(Canvas)

subclass นี้ใช้เพื่อเล่นเกม TicTacToe แบบโต้ตอบใน Jupyter notebooks class TicTacToe ถูกเรียกขณะเริ่มต้น subclass นี้

มาทำแมตช์ระหว่าง `random_player` และ `alphabeta_player` คลิกที่กระดานเพื่อเรียกผู้เล่นให้ทำการเดิน

In [ ]:
from notebook import Canvas_TicTacToe

In [ ]:
bot_play = Canvas_TicTacToe('bot_play', 'random', 'alphabeta')

ตอนนี้ มาเล่นเกมด้วยตัวเราเองกับ `random_player`:

In [ ]:
rand_play = Canvas_TicTacToe('rand_play', 'human', 'random')

ยี่ห้า! เรา (โดยปกติ) ชนะ แต่เราไม่สามารถชนะ `alphabeta_player` ได้ ไม่ว่าเราจะพยายามหนักแค่ไหน

In [ ]:
ab_play = Canvas_TicTacToe('ab_play', 'human', 'alphabeta')

In [ ]:
# สร้างเกม TicTacToe แบบอินเตอร์แอคทีฟอย่างง่าย
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import random

class SimpleTicTacToe:
    def __init__(self):
        self.board = [' ' for _ in range(9)]
        self.current_player = 'X'  # ผู้เล่นคือ X
        self.game_over = False
        self.winner = None
        
    def make_move(self, position):
        if self.board[position] == ' ' and not self.game_over:
            self.board[position] = self.current_player
            
            if self.check_winner():
                self.game_over = True
                self.winner = self.current_player
            elif ' ' not in self.board:
                self.game_over = True
                self.winner = 'Draw'
            else:
                self.current_player = 'O' if self.current_player == 'X' else 'X'
                # ถ้าเป็นคิว AI (O) ให้เดินอัตโนมัติ
                if self.current_player == 'O' and not self.game_over:
                    self.ai_move()
                    
    def ai_move(self):
        # AI เดินแบบสุ่ม
        available_moves = [i for i, spot in enumerate(self.board) if spot == ' ']
        if available_moves:
            move = random.choice(available_moves)
            self.board[move] = 'O'
            
            if self.check_winner():
                self.game_over = True
                self.winner = 'O'
            elif ' ' not in self.board:
                self.game_over = True
                self.winner = 'Draw'
            else:
                self.current_player = 'X'
    
    def check_winner(self):
        winning_combinations = [
            [0, 1, 2], [3, 4, 5], [6, 7, 8],  # แถว
            [0, 3, 6], [1, 4, 7], [2, 5, 8],  # คอลัมน์
            [0, 4, 8], [2, 4, 6]              # แนวทแยง
        ]
        
        for combo in winning_combinations:
            if (self.board[combo[0]] == self.board[combo[1]] == 
                self.board[combo[2]] != ' '):
                return True
        return False
    
    def display_board(self):
        board_html = """
        <style>
        .tic-tac-toe {
            display: grid;
            grid-template-columns: repeat(3, 80px);
            grid-template-rows: repeat(3, 80px);
            gap: 2px;
            margin: 20px 0;
        }
        .cell {
            background-color: #f0f0f0;
            border: 2px solid #333;
            display: flex;
            justify-content: center;
            align-items: center;
            font-size: 30px;
            font-weight: bold;
            cursor: pointer;
        }
        .cell:hover {
            background-color: #e0e0e0;
        }
        .cell.x {
            color: blue;
        }
        .cell.o {
            color: red;
        }
        .status {
            margin: 10px 0;
            font-size: 18px;
            font-weight: bold;
        }
        </style>
        """
        
        board_html += '<div class="tic-tac-toe">'
        for i in range(9):
            cell_class = "cell"
            if self.board[i] == 'X':
                cell_class += " x"
            elif self.board[i] == 'O':
                cell_class += " o"
                
            board_html += f'<div class="{cell_class}" onclick="makeMove({i})">{self.board[i]}</div>'
        board_html += '</div>'
        
        if self.game_over:
            if self.winner == 'Draw':
                status = "เกมเสมอ! 🤝"
            elif self.winner == 'X':
                status = "คุณชนะ! 🎉"
            else:
                status = "AI ชนะ! 🤖"
        else:
            status = f"คิวของ: {'คุณ (X)' if self.current_player == 'X' else 'AI (O)'}"
            
        board_html += f'<div class="status">{status}</div>'
        
        if self.game_over:
            board_html += '<button onclick="resetGame()">เริ่มเกมใหม่</button>'
        
        return HTML(board_html)

# สร้างเกมใหม่
game = SimpleTicTacToe()
display(game.display_board())

In [ ]:
# เพิ่ม JavaScript เพื่อจัดการการโต้ตอบ
from IPython.display import Javascript

js_code = """
window.currentGame = null;

function makeMove(position) {
    if (window.currentGame && !window.currentGame.game_over) {
        window.currentGame.make_move(position);
        updateDisplay();
    }
}

function resetGame() {
    window.currentGame = new SimpleTicTacToe();
    updateDisplay();
}

function updateDisplay() {
    // อัปเดตการแสดงผล
    var cells = document.querySelectorAll('.cell');
    cells.forEach((cell, index) => {
        cell.textContent = window.currentGame.board[index];
        cell.className = 'cell';
        if (window.currentGame.board[index] === 'X') {
            cell.className += ' x';
        } else if (window.currentGame.board[index] === 'O') {
            cell.className += ' o';
        }
    });
    
    var status = document.querySelector('.status');
    if (window.currentGame.game_over) {
        if (window.currentGame.winner === 'Draw') {
            status.textContent = 'เกมเสมอ! 🤝';
        } else if (window.currentGame.winner === 'X') {
            status.textContent = 'คุณชนะ! 🎉';
        } else {
            status.textContent = 'AI ชนะ! 🤖';
        }
    } else {
        status.textContent = 'คิวของ: ' + (window.currentGame.current_player === 'X' ? 'คุณ (X)' : 'AI (O)');
    }
}

// เริ่มต้นเกม
window.currentGame = {
    board: [' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' '],
    current_player: 'X',
    game_over: false,
    winner: null,
    
    make_move: function(position) {
        if (this.board[position] === ' ' && !this.game_over) {
            this.board[position] = this.current_player;
            
            if (this.check_winner()) {
                this.game_over = true;
                this.winner = this.current_player;
            } else if (!this.board.includes(' ')) {
                this.game_over = true;
                this.winner = 'Draw';
            } else {
                this.current_player = this.current_player === 'X' ? 'O' : 'X';
                // AI move
                if (this.current_player === 'O' && !this.game_over) {
                    setTimeout(() => this.ai_move(), 500);
                }
            }
        }
    },
    
    ai_move: function() {
        var available = [];
        for (var i = 0; i < 9; i++) {
            if (this.board[i] === ' ') available.push(i);
        }
        if (available.length > 0) {
            var move = available[Math.floor(Math.random() * available.length)];
            this.board[move] = 'O';
            
            if (this.check_winner()) {
                this.game_over = true;
                this.winner = 'O';
            } else if (!this.board.includes(' ')) {
                this.game_over = true;
                this.winner = 'Draw';
            } else {
                this.current_player = 'X';
            }
            updateDisplay();
        }
    },
    
    check_winner: function() {
        var wins = [
            [0,1,2], [3,4,5], [6,7,8],
            [0,3,6], [1,4,7], [2,5,8],
            [0,4,8], [2,4,6]
        ];
        
        for (var i = 0; i < wins.length; i++) {
            var [a, b, c] = wins[i];
            if (this.board[a] !== ' ' && 
                this.board[a] === this.board[b] && 
                this.board[b] === this.board[c]) {
                return true;
            }
        }
        return false;
    }
};
"""

display(Javascript(js_code))

In [ ]:
# เกม TicTacToe แบบอินเตอร์แอคทีฟ
from IPython.display import HTML

interactive_game_html = """
<div id="interactive-tictactoe">
    <style>
        .game-container {
            text-align: center;
            font-family: Arial, sans-serif;
            max-width: 400px;
            margin: 0 auto;
        }
        .game-board {
            display: grid;
            grid-template-columns: repeat(3, 100px);
            grid-template-rows: repeat(3, 100px);
            gap: 3px;
            margin: 20px auto;
            background-color: #333;
            padding: 10px;
            border-radius: 10px;
        }
        .game-cell {
            background-color: #fff;
            border: none;
            font-size: 36px;
            font-weight: bold;
            cursor: pointer;
            border-radius: 5px;
            transition: background-color 0.3s;
        }
        .game-cell:hover:not(:disabled) {
            background-color: #e8e8e8;
        }
        .game-cell:disabled {
            cursor: not-allowed;
        }
        .player-x {
            color: #2196F3;
        }
        .player-o {
            color: #FF5722;
        }
        .game-status {
            font-size: 18px;
            font-weight: bold;
            margin: 20px 0;
            min-height: 25px;
        }
        .reset-btn {
            background-color: #4CAF50;
            color: white;
            border: none;
            padding: 10px 20px;
            font-size: 16px;
            border-radius: 5px;
            cursor: pointer;
            margin-top: 10px;
        }
        .reset-btn:hover {
            background-color: #45a049;
        }
    </style>
    
    <div class="game-container">
        <h3>🎮 เกม Tic-Tac-Toe แบบอินเตอร์แอคทีฟ</h3>
        <p>คุณเป็น X, AI เป็น O</p>
        
        <div class="game-board">
            <button class="game-cell" onclick="makeGameMove(0)" id="cell-0"></button>
            <button class="game-cell" onclick="makeGameMove(1)" id="cell-1"></button>
            <button class="game-cell" onclick="makeGameMove(2)" id="cell-2"></button>
            <button class="game-cell" onclick="makeGameMove(3)" id="cell-3"></button>
            <button class="game-cell" onclick="makeGameMove(4)" id="cell-4"></button>
            <button class="game-cell" onclick="makeGameMove(5)" id="cell-5"></button>
            <button class="game-cell" onclick="makeGameMove(6)" id="cell-6"></button>
            <button class="game-cell" onclick="makeGameMove(7)" id="cell-7"></button>
            <button class="game-cell" onclick="makeGameMove(8)" id="cell-8"></button>
        </div>
        
        <div class="game-status" id="game-status">คิวของคุณ (X)</div>
        <button class="reset-btn" onclick="resetInteractiveGame()">🔄 เริ่มเกมใหม่</button>
    </div>
    
    <script>
        // ตัวแปรเกม
        let gameBoard = ['', '', '', '', '', '', '', '', ''];
        let currentPlayer = 'X';
        let gameActive = true;
        
        const winningConditions = [
            [0, 1, 2], [3, 4, 5], [6, 7, 8],
            [0, 3, 6], [1, 4, 7], [2, 5, 8],
            [0, 4, 8], [2, 4, 6]
        ];
        
        function makeGameMove(cellIndex) {
            if (gameBoard[cellIndex] !== '' || !gameActive || currentPlayer !== 'X') {
                return;
            }
            
            // ผู้เล่นเดิน
            gameBoard[cellIndex] = 'X';
            updateGameDisplay();
            
            if (checkGameWinner()) {
                return;
            }
            
            if (gameBoard.every(cell => cell !== '')) {
                document.getElementById('game-status').textContent = 'เกมเสมอ! 🤝';
                gameActive = false;
                return;
            }
            
            // AI เดิน
            currentPlayer = 'O';
            document.getElementById('game-status').textContent = 'AI กำลังคิด...';
            
            setTimeout(() => {
                makeAIMove();
                updateGameDisplay();
                
                if (checkGameWinner()) {
                    return;
                }
                
                if (gameBoard.every(cell => cell !== '')) {
                    document.getElementById('game-status').textContent = 'เกมเสมอ! 🤝';
                    gameActive = false;
                    return;
                }
                
                currentPlayer = 'X';
                document.getElementById('game-status').textContent = 'คิวของคุณ (X)';
            }, 1000);
        }
        
        function makeAIMove() {
            // AI แบบง่าย - เลือกช่องว่างแบบสุ่ม
            const availableCells = gameBoard.map((cell, index) => cell === '' ? index : null)
                                           .filter(val => val !== null);
            
            if (availableCells.length > 0) {
                const randomIndex = Math.floor(Math.random() * availableCells.length);
                const aiChoice = availableCells[randomIndex];
                gameBoard[aiChoice] = 'O';
            }
        }
        
        function updateGameDisplay() {
            for (let i = 0; i < 9; i++) {
                const cell = document.getElementById(`cell-${i}`);
                cell.textContent = gameBoard[i];
                cell.className = 'game-cell';
                
                if (gameBoard[i] === 'X') {
                    cell.className += ' player-x';
                    cell.disabled = true;
                } else if (gameBoard[i] === 'O') {
                    cell.className += ' player-o';
                    cell.disabled = true;
                } else {
                    cell.disabled = false;
                }
            }
        }
        
        function checkGameWinner() {
            for (let condition of winningConditions) {
                const [a, b, c] = condition;
                
                if (gameBoard[a] && gameBoard[a] === gameBoard[b] && gameBoard[a] === gameBoard[c]) {
                    if (gameBoard[a] === 'X') {
                        document.getElementById('game-status').textContent = 'คุณชนะ! 🎉';
                    } else {
                        document.getElementById('game-status').textContent = 'AI ชนะ! 🤖';
                    }
                    
                    gameActive = false;
                    
                    // เน้นเซลล์ที่ชนะ
                    condition.forEach(index => {
                        document.getElementById(`cell-${index}`).style.backgroundColor = '#ffeb3b';
                    });
                    
                    return true;
                }
            }
            return false;
        }
        
        function resetInteractiveGame() {
            gameBoard = ['', '', '', '', '', '', '', '', ''];
            currentPlayer = 'X';
            gameActive = true;
            document.getElementById('game-status').textContent = 'คิวของคุณ (X)';
            
            for (let i = 0; i < 9; i++) {
                const cell = document.getElementById(`cell-${i}`);
                cell.textContent = '';
                cell.disabled = false;
                cell.className = 'game-cell';
                cell.style.backgroundColor = '';
            }
        }
        
        // เริ่มต้นเกม
        updateGameDisplay();
    </script>
</div>
"""

display(HTML(interactive_game_html))

In [ ]:
# เกม TicTacToe พร้อม AlphaBeta AI (เวอร์ชันที่ใช้งานได้)
from IPython.display import HTML
import ipywidgets as widgets
from IPython.display import display, clear_output

class InteractiveTicTacToeAI:
    def __init__(self):
        self.ttt = TicTacToe()
        self.game_state = self.ttt.initial
        self.buttons = []
        self.status_widget = None
        self.game_container = None
        self.setup_game()
        
    def setup_game(self):
        """สร้าง UI สำหรับเกม"""
        # สร้างปุ่มสำหรับกระดาน 3x3
        self.buttons = []
        for i in range(3):
            row = []
            for j in range(3):
                btn = widgets.Button(
                    description='',
                    layout=widgets.Layout(width='80px', height='80px'),
                    style=widgets.ButtonStyle(
                        font_size='24px',
                        font_weight='bold'
                    )
                )
                # เก็บตำแหน่งในปุ่ม
                btn.row = i + 1  # TicTacToe ใช้ 1-based indexing
                btn.col = j + 1
                btn.on_click(self.on_button_click)
                row.append(btn)
            self.buttons.append(row)
        
        # สถานะเกม
        self.status_widget = widgets.HTML(
            value="<h3 style='color: blue;'>🎮 คิวของคุณ (X)</h3>"
        )
        
        # ปุ่มรีเซ็ต
        reset_btn = widgets.Button(
            description='🔄 เริ่มเกมใหม่',
            button_style='success',
            layout=widgets.Layout(width='150px')
        )
        reset_btn.on_click(self.reset_game)
        
        # จัดเรียง UI
        board_rows = []
        for row in self.buttons:
            board_rows.append(widgets.HBox(row))
        
        self.game_container = widgets.VBox([
            widgets.HTML("<h2 style='text-align: center;'>🧠 TicTacToe vs AlphaBeta AI</h2>"),
            widgets.HTML("<p style='text-align: center;'>คุณ: X (น้ำเงิน) | AI: O (แดง)</p>"),
            self.status_widget,
            widgets.VBox(board_rows, layout=widgets.Layout(align_items='center')),
            widgets.HTML("<br>"),
            widgets.HBox([reset_btn], layout=widgets.Layout(justify_content='center'))
        ])
        
        self.update_board_display()
        
    def on_button_click(self, button):
        """จัดการเมื่อคลิกปุ่ม"""
        if self.game_state.to_move != 'X':
            return
            
        move = (button.row, button.col)
        
        if move in self.game_state.moves:
            # ผู้เล่นเดิน
            self.game_state = self.ttt.result(self.game_state, move)
            self.update_board_display()
            
            # ตรวจสอบการจบเกม
            if self.ttt.terminal_test(self.game_state):
                self.handle_game_over()
                return
            
            # AI เดิน
            if self.game_state.to_move == 'O':
                self.status_widget.value = "<h3 style='color: orange;'>🤖 AI กำลังคิด...</h3>"
                
                # ให้ AI เดิน
                ai_move = alphabeta_player(self.ttt, self.game_state)
                self.game_state = self.ttt.result(self.game_state, ai_move)
                
                self.update_board_display()
                
                # ตรวจสอบการจบเกมหลัง AI เดิน
                if self.ttt.terminal_test(self.game_state):
                    self.handle_game_over()
                else:
                    self.status_widget.value = "<h3 style='color: blue;'>🎮 คิวของคุณ (X)</h3>"
    
    def update_board_display(self):
        """อัปเดตการแสดงผลกระดาน"""
        # เคลียร์กระดาน
        for i in range(3):
            for j in range(3):
                self.buttons[i][j].description = ''
                self.buttons[i][j].button_style = ''
                self.buttons[i][j].disabled = False
        
        # แสดงเครื่องหมายบนกระดาน
        for (row, col), player in self.game_state.board.items():
            if 1 <= row <= 3 and 1 <= col <= 3:
                btn = self.buttons[row-1][col-1]
                btn.description = player
                btn.disabled = True
                if player == 'X':
                    btn.button_style = 'info'  # สีน้ำเงิน
                else:
                    btn.button_style = 'danger'  # สีแดง
    
    def handle_game_over(self):
        """จัดการเมื่อเกมจบ"""
        utility = self.ttt.utility(self.game_state, 'X')
        
        if utility == 1:
            self.status_widget.value = "<h3 style='color: green;'>🎉 คุณชนะ! เก่งมาก!</h3>"
        elif utility == -1:
            self.status_widget.value = "<h3 style='color: red;'>🤖 AI ชนะ! ลองใหม่นะ</h3>"
        else:
            self.status_widget.value = "<h3 style='color: purple;'>🤝 เสมอ! เล่นได้ดีมาก!</h3>"
        
        # ปิดปุ่มทั้งหมด
        for i in range(3):
            for j in range(3):
                self.buttons[i][j].disabled = True
    
    def reset_game(self, button=None):
        """รีเซ็ตเกม"""
        self.game_state = self.ttt.initial
        self.update_board_display()
        self.status_widget.value = "<h3 style='color: blue;'>🎮 คิวของคุณ (X)</h3>"
    
    def display(self):
        """แสดงเกม"""
        return self.game_container

# สร้างและแสดงเกม
ai_game = InteractiveTicTacToeAI()
display(ai_game.display())

In [4]:
from collections import namedtuple
import ipywidgets as widgets
from IPython.display import display, clear_output

# โครงสร้างสถานะของเกม
GameState = namedtuple('GameState', 'to_move, utility, board, moves')

# 1. คลาสจัดการกติกาเกม Hexapawn
class Hexapawn:
    def __init__(self):
        # กระดานเริ่มต้น (X อยู่แถว 3 เดินขึ้น, O อยู่แถว 1 เดินลง)
        board = {
            (3, 1): 'X', (3, 2): 'X', (3, 3): 'X',
            (1, 1): 'O', (1, 2): 'O', (1, 3): 'O'
        }
        self.initial = GameState(to_move='X', utility=0, board=board, 
                                 moves=self.compute_moves(board, 'X'))

    def compute_moves(self, board, player):
        """คำนวณตาเดินทั้งหมดที่เป็นไปได้"""
        moves = []
        direction = -1 if player == 'X' else 1  # X เดินขึ้น(-1) O เดินลง(+1)
        opponent = 'O' if player == 'X' else 'X'

        for (r, c), p in board.items():
            if p == player:
                # 1. เดินหน้าตรง (ถ้าช่องว่าง)
                forward = (r + direction, c)
                if forward not in board and 1 <= forward[0] <= 3:
                    moves.append(((r, c), forward))
                
                # 2. กินเฉียง (ถ้ามีหมากศัตรู)
                for dc in [-1, 1]:
                    diagonal = (r + direction, c + dc)
                    if diagonal in board and board[diagonal] == opponent:
                        moves.append(((r, c), diagonal))
        return moves

    def actions(self, state):
        return state.moves

    def result(self, state, move):
        """อัปเดตกระดานและเช็คสถานะการจบเกมหลังจากมีการเดิน"""
        start, end = move
        board = state.board.copy()
        player = state.to_move

        # เดินหมาก
        board[end] = player
        del board[start]

        next_player = 'O' if player == 'X' else 'X'
        next_moves = self.compute_moves(board, next_player)

        # เช็คเงื่อนไขชนะ
        end_row = 1 if player == 'X' else 3
        if end[0] == end_row:  # เดินถึงขอบกระดาน
            util = 1 if player == 'X' else -1
        elif not next_moves:   # ศัตรูขยับไม่ได้ (หรือโดนกินหมด)
            util = 1 if player == 'X' else -1
        else:
            util = 0

        return GameState(to_move=next_player, utility=util, board=board, moves=next_moves)

    def utility(self, state, player):
        return state.utility if player == 'X' else -state.utility

    def terminal_test(self, state):
        return state.utility != 0 or len(state.moves) == 0

# 2. อัลกอริทึม Alpha-Beta Pruning
def alphabeta_search(state, game):
    player = state.to_move

    def max_value(state, alpha, beta):
        if game.terminal_test(state):
            return game.utility(state, player)
        v = -float('inf')
        for a in game.actions(state):
            v = max(v, min_value(game.result(state, a), alpha, beta))
            if v >= beta:
                return v
            alpha = max(alpha, v)
        return v

    def min_value(state, alpha, beta):
        if game.terminal_test(state):
            return game.utility(state, player)
        v = float('inf')
        for a in game.actions(state):
            v = min(v, max_value(game.result(state, a), alpha, beta))
            if v <= alpha:
                return v
            beta = min(beta, v)
        return v

    best_score = -float('inf')
    beta = float('inf')
    best_action = None
    for a in game.actions(state):
        v = min_value(game.result(state, a), best_score, beta)
        if v > best_score:
            best_score = v
            best_action = a
    return best_action

def alphabeta_player(game, state):
    return alphabeta_search(state, game)

# 3. User Interface (UI) สำหรับ Hexapawn
class InteractiveHexapawnAI:
    def __init__(self):
        self.game = Hexapawn()
        self.game_state = self.game.initial
        self.selected_pos = None  # เก็บตำแหน่งหมากที่ถูกคลิกเลือก
        self.buttons = []
        self.status_widget = None
        self.game_container = None
        self.setup_game()
        
    def setup_game(self):
        self.buttons = []
        for i in range(3):
            row = []
            for j in range(3):
                btn = widgets.Button(
                    description='',
                    layout=widgets.Layout(width='80px', height='80px'),
                    style=widgets.ButtonStyle(font_size='24px', font_weight='bold')
                )
                btn.row = i + 1
                btn.col = j + 1
                btn.on_click(self.on_button_click)
                row.append(btn)
            self.buttons.append(row)
        
        self.status_widget = widgets.HTML(
            value="<h3 style='color: blue;'>🎮 คิวของคุณ (X) - เลือกหมากเพื่อเดิน</h3>"
        )
        
        reset_btn = widgets.Button(
            description='🔄 เริ่มเกมใหม่',
            button_style='success',
            layout=widgets.Layout(width='150px')
        )
        reset_btn.on_click(self.reset_game)
        
        board_rows = [widgets.HBox(row) for row in self.buttons]
        
        self.game_container = widgets.VBox([
            widgets.HTML("<h2 style='text-align: center;'>🧠 Hexapawn vs AlphaBeta AI</h2>"),
            widgets.HTML("<p style='text-align: center;'>คุณ: X (เดินขึ้น) | AI: O (เดินลง)</p>"),
            self.status_widget,
            widgets.VBox(board_rows, layout=widgets.Layout(align_items='center')),
            widgets.HTML("<br>"),
            widgets.HBox([reset_btn], layout=widgets.Layout(justify_content='center'))
        ])
        
        self.update_board_display()
        
    def on_button_click(self, button):
        if self.game_state.to_move != 'X':
            return
            
        clicked_pos = (button.row, button.col)
        
        # 1. ผู้เล่นยังไม่ได้เลือกหมาก
        if self.selected_pos is None:
            if clicked_pos in self.game_state.board and self.game_state.board[clicked_pos] == 'X':
                self.selected_pos = clicked_pos
                self.update_board_display()
                self.status_widget.value = "<h3 style='color: blue;'>📍 เลือกช่องที่จะเดิน...</h3>"
                
        # 2. ผู้เล่นเลือกหมากไว้แล้ว
        else:
            # ถ้าคลิกตัวเดิม (ยกเลิกการเลือก)
            if clicked_pos == self.selected_pos:
                self.selected_pos = None
                self.update_board_display()
                self.status_widget.value = "<h3 style='color: blue;'>🎮 คิวของคุณ (X) - เลือกหมากเพื่อเดิน</h3>"
            
            # ถ้าคลิกหมากตัวอื่นของตัวเอง (เปลี่ยนตัว)
            elif clicked_pos in self.game_state.board and self.game_state.board[clicked_pos] == 'X':
                self.selected_pos = clicked_pos
                self.update_board_display()
                
            # พยายามเดินไปยังช่องที่เลือก
            else:
                move = (self.selected_pos, clicked_pos)
                if move in self.game_state.moves:
                    self.selected_pos = None
                    self.game_state = self.game.result(self.game_state, move)
                    self.update_board_display()
                    
                    if self.game.terminal_test(self.game_state):
                        self.handle_game_over()
                        return
                    
                    # ตาของ AI
                    self.status_widget.value = "<h3 style='color: orange;'>🤖 AI กำลังคิด...</h3>"
                    ai_move = alphabeta_player(self.game, self.game_state)
                    self.game_state = self.game.result(self.game_state, ai_move)
                    self.update_board_display()
                    
                    if self.game.terminal_test(self.game_state):
                        self.handle_game_over()
                    else:
                        self.status_widget.value = "<h3 style='color: blue;'>🎮 คิวของคุณ (X) - เลือกหมากเพื่อเดิน</h3>"
    
    def update_board_display(self):
        for i in range(3):
            for j in range(3):
                btn = self.buttons[i][j]
                pos = (i + 1, j + 1)
                
                # ล้างการแสดงผล
                btn.description = ''
                btn.button_style = ''
                btn.disabled = False
                
                # วาดหมากใหม่
                if pos in self.game_state.board:
                    player = self.game_state.board[pos]
                    btn.description = player
                    if player == 'X':
                        # ไฮไลต์ถ้าหมากกำลังถูกเลือก (สีเหลือง/warning)
                        btn.button_style = 'warning' if pos == self.selected_pos else 'info'
                    else:
                        btn.button_style = 'danger'
    
    def handle_game_over(self):
        utility = self.game.initial.utility # Helper ดึงค่า
        actual_utility = self.game_state.utility
        
        if actual_utility == 1:
            self.status_widget.value = "<h3 style='color: green;'>🎉 คุณชนะ!</h3>"
        elif actual_utility == -1:
            self.status_widget.value = "<h3 style='color: red;'>🤖 AI ชนะ!</h3>"
            
        for i in range(3):
            for j in range(3):
                self.buttons[i][j].disabled = True
                
    def reset_game(self, button=None):
        self.game_state = self.game.initial
        self.selected_pos = None
        self.update_board_display()
        self.status_widget.value = "<h3 style='color: blue;'>🎮 คิวของคุณ (X) - เลือกหมากเพื่อเดิน</h3>"
        
    def display(self):
        return self.game_container

# รันเกม
ai_game = InteractiveHexapawnAI()
display(ai_game.display())